# kNN

### Overview

- [Setup and Data Loading](#Setup-and-Data-Loading)
- [Data Understanding](#Data-Understanding)
- [Encoding the Features](#Encoding-the-Features)
- [The Majority Baseline](#The-Majority-Baseline)
- [A First kNN Model](#A-First-kNN-Model)
- [Tuning kNN](#Tuning-kNN)
- [Final Evaluation and the Imbalance Effect](#Final-Evaluation-and-the-Imbalance-Effect)
- [Saving the Results](#Saving-the-Results)

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix, classification_report

## Setup and Data Loading

The train/test split and the leakage-safe publisher_tier are already created in Phase B.

In [2]:
SCOPE = "scope_2000_2018"
DATA_DIR = "data/processed/" + SCOPE

X_train = pd.read_csv(DATA_DIR + "/X_train.csv")
X_test  = pd.read_csv(DATA_DIR + "/X_test.csv")
y_train = pd.read_csv(DATA_DIR + "/y_train.csv")["Hit"]
y_test  = pd.read_csv(DATA_DIR + "/y_test.csv")["Hit"]

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)
X_train.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/scope_2000_2018/X_train.csv'

## Data Understanding

The model only gets three pre-release features:

| Feature | Type | Meaning |
|---|---|---|
| `genre` | nominal | game genre |
| `platform_family` | nominal | Nintendo, PC, PlayStation, Xbox, Other |
| `publisher_tier` | ordinal | publisher's past million-seller count |

The target is imbalanced, so accuracy alone is not enough.

In [ ]:
print('Hit  (1):', len(y_train[y_train == 1]) / len(y_train))
print('Flop (0):', len(y_train[y_train == 0]) / len(y_train))

print()
print('Training labels:')
print(y_train.value_counts().sort_index())

## Encoding the Features

For kNN I use one-hot encoding. This avoids giving the nominal categories an artificial order.

After one-hot encoding, I scale the data. This is important for kNN because the algorithm compares distances between records.

In [ ]:
X_train_enc = pd.get_dummies(X_train.astype(str))
X_test_enc  = pd.get_dummies(X_test.astype(str))

# make sure train and test have the same dummy columns
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enc)
X_test_scaled  = scaler.transform(X_test_enc)

print('Number of features after one-hot encoding:', X_train_enc.shape[1])
X_train_enc.head()

## The Majority Baseline

A simple baseline always predicts the majority class, which is Flop.

In [ ]:
baseline_pred = np.zeros(len(y_test), dtype=int)

print('Baseline accuracy:', accuracy_score(y_test, baseline_pred))
print('Baseline F1      :', f1_score(y_test, baseline_pred, zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(y_test, baseline_pred))

## A First kNN Model

First I try one simple kNN model. Like in the lecture, k controls how local or smooth the decision is.

In [ ]:
knn_first = KNeighborsClassifier(n_neighbors=7, metric='euclidean', weights='uniform')
knn_first.fit(X_train_scaled, y_train)

pred = knn_first.predict(X_test_scaled)
proba = knn_first.predict_proba(X_test_scaled)[:, 1]

print('Test accuracy:', accuracy_score(y_test, pred))
print('Test F1      :', f1_score(y_test, pred))
print('Test ROC-AUC :', roc_auc_score(y_test, proba))
print('Confusion matrix:')
print(confusion_matrix(y_test, pred))

## Tuning kNN

Now I tune a small parameter grid. I use ROC-AUC for the cross-validation score because the project compares models mainly with ROC-AUC and F1.

In [ ]:
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 15, 21, 31, 41, 51],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn = KNeighborsClassifier()
knn_cv = GridSearchCV(knn, param_grid, scoring='roc_auc', cv=5, n_jobs=-1)
knn_cv.fit(X_train_scaled, y_train)

print('Best CV ROC-AUC:', knn_cv.best_score_)
print('Best parameters:', knn_cv.best_params_)

In [ ]:
results = pd.DataFrame(knn_cv.cv_results_)
plot_data = results[results['param_weights'] == 'distance'].copy()
plot_data = plot_data.groupby('param_n_neighbors')['mean_test_score'].max()

plt.figure(figsize=(7, 4))
plt.plot(plot_data.index, plot_data.values, marker='o')
plt.xlabel('k')
plt.ylabel('CV ROC-AUC')
plt.title('kNN tuning: effect of k')
plt.grid(True, alpha=0.3)
plt.show()

## Final Evaluation and the Imbalance Effect

The final comparison uses the same tuned procedure twice:

1. train on the original imbalanced training set
2. train on a 50/50 down-sampled training set

The test set is not changed.

In [ ]:
def evaluate(model, X_te, y_te, model_name, resample):
    proba = model.predict_proba(X_te)[:, 1]
    pred = model.predict(X_te)

    roc = roc_auc_score(y_te, proba)
    f1 = f1_score(y_te, pred)
    acc = accuracy_score(y_te, pred)

    print('--- %s (resample=%s) ---' % (model_name, resample))
    print('ROC-AUC : %.3f' % roc)
    print('F1      : %.3f' % f1)
    print('Accuracy: %.3f' % acc)
    print('Confusion matrix [rows = true 0/1, cols = predicted 0/1]:')
    print(confusion_matrix(y_te, pred))
    print(classification_report(y_te, pred, zero_division=0))

    return {'model': model_name, 'resample': resample,
            'roc_auc': roc, 'f1': f1, 'accuracy': acc}


def fit_knn(X_tr, y_tr):
    gs = GridSearchCV(KNeighborsClassifier(), param_grid,
                      scoring='roc_auc', cv=5, n_jobs=-1, refit=True)
    gs.fit(X_tr, y_tr)
    print('best parameters:', gs.best_params_)
    return gs.best_estimator_

### Run 1 — no resampling

In [ ]:
knn_none = fit_knn(X_train_scaled, y_train)
row_none = evaluate(knn_none, X_test_scaled, y_test, 'knn', 'none')

### Run 2 — 50/50 down-sampling

In [ ]:
train = X_train_enc.copy()
train['Hit'] = y_train.values

hits = train[train['Hit'] == 1]
flops = train[train['Hit'] == 0]

flops_down = flops.sample(n=len(hits), random_state=0)
train_bal = pd.concat([hits, flops_down]).sample(frac=1, random_state=0)

X_train_bal = train_bal.drop(columns='Hit')
y_train_bal = train_bal['Hit']

scaler_bal = StandardScaler()
X_train_bal_scaled = scaler_bal.fit_transform(X_train_bal)
X_test_bal_scaled = scaler_bal.transform(X_test_enc)

print('Balanced training set size:', len(y_train_bal))
print('Hit  (1):', len(y_train_bal[y_train_bal == 1]))
print('Flop (0):', len(y_train_bal[y_train_bal == 0]))

In [ ]:
knn_down = fit_knn(X_train_bal_scaled, y_train_bal)
row_down = evaluate(knn_down, X_test_bal_scaled, y_test, 'knn', 'downsample')

## Saving the Results

The two runs are saved with the same columns as the other Phase C model notebooks.

In [ ]:
os.makedirs('results', exist_ok=True)

results = pd.DataFrame([row_none, row_down])
results = results[['model', 'resample', 'roc_auc', 'f1', 'accuracy']]

out_path = 'results/knn.csv'
results.to_csv(out_path, index=False)
print('Saved', out_path)
results